In [2]:
# positional encoding 

import torch 
import torch.nn as nn 
import torch.nn.functional as F


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self , max_length , emb_dim , dropout = 0.1):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(max_length , emb_dim) *0.02)
        self.dropout = nn.Dropout(dropout)
    def forward(self , x):
        x = x + self.pos_emb[:x.size(1)]
        return self.dropout(x)


In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self , emb_dim , num_heads , dropout = 0.1):
        super().__init__()
        self.h = num_heads
        self.d = emb_dim // num_heads
        self.q_proj = nn.Linear(emb_dim , emb_dim)
        self.k_proj = nn.Linear(emb_dim , emb_dim)
        self.v_proj = nn.Linear(emb_dim , emb_dim)
        self.out_proj = nn.Linear(emb_dim , emb_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self , x):
        return x.view(x.size(0) , x.size(1) , self.h , self.d).transpose(1 , 2)
    def forward(self , query , key , value , attn_mask = None , key_padding_mask = None):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        scores = q @ k.transpose(2, 3) / (self.d ** 0.5)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask , -torch.inf)
        if key_padding_mask is not None:
            mask = mask.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2) )
            scores = scores.masked_fill(mask , -torch.inf)
        weights = F.softmax(scores , dim = -1)
        z= self.dropout(weights) @ v
        z = z.transpose(1 , 2)
        z= z.reshape(z.size(0) , z.size(1) , self.h * self.d)
        return (self.out_proj(z),weights)

      

In [7]:
class TransformerEncodingLayer(nn.Module):
    def __init__(self , d_model , num_heads , dim_feedforward=2048 , dropout = 0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model , num_heads , dropout)
        self.Leaner1 = nn.Linear(d_model , dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.Leaner2 = nn.Linear(dim_feedforward , d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self , src , src_mask = None , src_key_padding_mask = None):
        attn , _ = self.self_attn(src , src , src , attn_mask = src_mask , key_padding_mask = src_key_padding_mask)
        z= self.norm1(src + self.dropout(attn))
        ff = self.dropout(self.Leaner2(self.dropout(F.relu(self.Leaner1(z)))))
        z= self.norm2(z + ff)
        return z
        

In [ ]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self , d_model , num_heads , dim_feedforward=2048 , dropout = 0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model , num_heads , dropout)
        self.multihead_attn = MultiheadAttention(d_model , num_heads , dropout)
        self.Leaner1 = nn.Linear(d_model , dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.Leaner2 = nn.Linear(dim_feedforward , d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self , tgt , memory , tgt_mask = None , memory_mask = None ,
                tgt_key_padding_mask = None , memory_key_padding_mask = None):
        attn1 , _ = self.self_attn(tgt , tgt , tgt , attn_mask = tgt_mask ,
                                   key_padding_mask = tgt_key_padding_mask)
        z= self.norm1(tgt + self.dropout(attn1))
        attn2 , _ = self.multihead_attn(z , memory , memory ,
                                        attn_mask = memory_mask ,
                                        key_padding_mask = memory_key_padding_mask)
        z= self.norm2(z + self.dropout(attn2))
        ff = self.dropout(self.Leaner2(self.dropout(F.relu(self.Leaner1(z)))))
        z= self.norm3(z + ff)
        return z